In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

!pip install \
    transformers \
    accelerate \
    bitsandbytes \
    sentencepiece \
    peft \
    rouge-score \
    evaluate \
    open_clip_torch \
    scikit-learn \
    nltk

!pip install faiss-cpu
!pip install torchxrayvision

Looking in indexes: https://download.pytorch.org/whl/cu124


In [2]:
# Imports all the necessary libraries
import os
import gc
import json
import time
import random
import pickle
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Optional

import faiss
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import densenet121
import torchxrayvision as xrv

from PIL import Image

from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer

import re
from sklearn.metrics import f1_score

from transformers import (
    TrainingArguments,
    Trainer,
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration,
)

from transformers.models.minicpmv4_6.modeling_minicpmv4_6 import (
    MiniCPMV4_6ForConditionalGeneration
)

warnings.filterwarnings("ignore")

import nltk

nltk.download("punkt")

f:\Mimic\mimic\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to C:\Users\INSTRUMENT
[nltk_data]     LAB\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
DATA_ROOT = Path("F:\Mimic\data")

VECTOR_DIR = DATA_ROOT / "vector_store"
DATA_DIR = DATA_ROOT
IMAGE_ROOT = Path("F:\Mimic\mimics\test_dataset\images")

VIEW_CLASSIFIER_PATH = (
    DATA_ROOT /
    "checkpoints" /
    "view_classifier.pt"
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Detected device: {DEVICE}")

ACTIVE_MODEL = "qwen2.5-vl-3b"
MAX_TEST_SAMPLES = 2
TOP_K = 5

AP_WEIGHT = 0.65
LV_WEIGHT = 0.35

Detected device: cuda


In [4]:
# ============================================================
# CONFIGURATION
# ============================================================

VLM_MODELS = {
    "qwen2.5-vl-3b": "Qwen/Qwen2.5-VL-3B-Instruct",
    "nv-reason-cxr-3b": "nvidia/NV-Reason-CXR-3B",
    "minicpm-v-4.6": "openbmb/MiniCPM-V-4.6",
}

In [5]:
def cleanup_cuda():

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def cosine_similarity(a, b):

    a = a / (np.linalg.norm(a) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)

    return np.dot(a, b)

In [6]:
# Load all embeddings and meta data
print("Loading metadata and embeddings...")

# ============================================================
# METADATA
# ============================================================

with open(DATA_DIR / "parsed_reports.json") as f:
    parsed_reports = json.load(f)

with open(DATA_DIR / "temporal_patients.json") as f:
    temporal_patients = json.load(f)

with open(DATA_DIR / "trajectories.json") as f:
    trajectories = json.load(f)

with open(DATA_DIR / "temporal_windows.json") as f:
    temporal_windows = json.load(f)

with open(DATA_DIR / "view_index.json") as f:
    view_index = json.load(f)

# ============================================================
# EMBEDDINGS
# ============================================================

report_embeddings = np.load(
    VECTOR_DIR / "reports_embeddings.npy"
)

trajectory_embeddings = np.load(
    VECTOR_DIR / "trajectories_embeddings.npy"
)

window_embeddings = np.load(
    VECTOR_DIR / "windows_embeddings.npy"
)

img_ap_embeddings = np.load(
    VECTOR_DIR / "img_ap_embeddings.npy"
)

img_lv_embeddings = np.load(
    VECTOR_DIR / "img_lv_embeddings.npy"
)

fused_embeddings = np.load(
    VECTOR_DIR / "fused_reports_embeddings.npy"
)

print("Metadata + embeddings loaded")

Loading metadata and embeddings...
Metadata + embeddings loaded


In [7]:
with open(VECTOR_DIR / "reports_meta.json") as f:

    reports_meta = json.load(f)

report_keys = reports_meta["keys"]

with open(VECTOR_DIR / "trajectories_meta.json") as f:

    trajectories_meta = json.load(f)

trajectory_keys = trajectories_meta["keys"]

with open(
    VECTOR_DIR / "trajectories_metadata.pkl",
    "rb"
) as f:

    trajectory_metadata = pickle.load(f)

with open(VECTOR_DIR / "windows_meta.json") as f:

    windows_meta = json.load(f)

window_keys = windows_meta["keys"]

with open(
    VECTOR_DIR / "windows_metadata.pkl",
    "rb"
) as f:

    window_metadata = pickle.load(f)

with open(VECTOR_DIR / "img_ap_meta.json") as f:

    ap_meta = json.load(f)

ap_keys = ap_meta["keys"]


with open(VECTOR_DIR / "img_lv_meta.json") as f:

    lv_meta = json.load(f)

lv_keys = lv_meta["keys"]

In [8]:
# Build All FAISS Indices
def build_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    normalized = np.ascontiguousarray(
        embeddings.astype(np.float32)
    )
    
    faiss.normalize_L2(normalized)
    index.add(normalized)
    return index


report_index = build_index(report_embeddings)
trajectory_index = build_index(trajectory_embeddings)
window_index = build_index(window_embeddings)
ap_index = build_index(img_ap_embeddings)
lv_index = build_index(img_lv_embeddings)
fused_index = build_index(fused_embeddings)

In [9]:
# Load BiomedCLIP + Query Encoding + Temporal Intent Detection
import open_clip

print("Loading BiomedCLIP...")

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
)

clip_tokenizer = open_clip.get_tokenizer(
    "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
)


# ============================================================
# CLIP DEVICE SELECTION (SAFE FALLBACK)
# ============================================================

CLIP_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if CLIP_DEVICE == "cuda":
    try:
        clip_model = clip_model.to("cuda")
        clip_model.eval()

        # Smoke test to catch unsupported CUDA kernel/runtime combos
        with torch.no_grad():
            _tokens = clip_tokenizer(["test"]).to("cuda")
            _ = clip_model.encode_text(_tokens)

        print("BiomedCLIP running on CUDA")

    except Exception as e:
        print(f"CUDA not usable for BiomedCLIP, falling back to CPU: {e}")
        CLIP_DEVICE = "cpu"
        clip_model = clip_model.to("cpu")
        clip_model.eval()
else:
    clip_model = clip_model.to("cpu")
    clip_model.eval()
    print("BiomedCLIP running on CPU")


# ============================================================
# QUERY ENCODERS
# ============================================================

def encode_text_query(text):

    with torch.no_grad():

        tokens = clip_tokenizer([text]).to(CLIP_DEVICE)

        emb = clip_model.encode_text(tokens)

        emb = F.normalize(emb, dim=-1)

    return emb.cpu().numpy()[0]


def encode_image_query(image_path):

    image = clip_preprocess(
        Image.open(image_path).convert("RGB")
    ).unsqueeze(0).to(CLIP_DEVICE)

    with torch.no_grad():

        emb = clip_model.encode_image(image)

        emb = F.normalize(emb, dim=-1)

    return emb.cpu().numpy()[0]

# ============================================================
# TEMPORAL INTENT DETECTION
# ============================================================

def detect_temporal_intent(text):

    text = text.lower()

    worsening_words = [
        "worsening",
        "progression",
        "increase",
        "larger",
        "worse",
    ]

    improving_words = [
        "improving",
        "resolved",
        "better",
        "decrease",
        "reduced",
    ]

    stable_words = [
        "stable",
        "unchanged",
        "persistent",
    ]

    if any(w in text for w in worsening_words):
        return "worsening"

    if any(w in text for w in improving_words):
        return "improving"

    if any(w in text for w in stable_words):
        return "stable"

    return "unknown"

Loading BiomedCLIP...


BiomedCLIP running on CUDA


In [10]:
# ============================================================
# VIEW CLASSIFIER
# ============================================================

class ViewClassifierModel(nn.Module):

    def __init__(self):

        super().__init__()

        # ====================================================
        # BACKBONE
        # ====================================================

        self.backbone = xrv.models.DenseNet(
            weights="densenet121-res224-mimic_ch"
        )

        # ====================================================
        # CLASSIFIER HEAD
        # ====================================================

        self.classifier = nn.Linear(
            1024,
            2
        )

    def forward(self, x):

        # ====================================================
        # FEATURE EXTRACTION
        # ====================================================

        feats = self.backbone.features(x)

        feats = F.relu(feats, inplace=True)

        feats = F.adaptive_avg_pool2d(
            feats,
            (1, 1)
        )

        feats = torch.flatten(
            feats,
            1
        )

        # ====================================================
        # CLASSIFICATION
        # ====================================================

        logits = self.classifier(feats)

        return logits


print("Loading view classifier...")

view_classifier = ViewClassifierModel()

checkpoint = torch.load(
    VIEW_CLASSIFIER_PATH,
    map_location=DEVICE,
)

# ============================================================
# HANDLE DIFFERENT CHECKPOINT FORMATS
# ============================================================

if "state_dict" in checkpoint:

    checkpoint = checkpoint["state_dict"]

view_classifier.load_state_dict(
    checkpoint,
    strict=False,
)

view_classifier = view_classifier.to(DEVICE)

view_classifier.eval()

print("View classifier loaded")

Loading view classifier...
View classifier loaded


In [11]:
view_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [12]:
def predict_view(image_path):

    image = Image.open(
        image_path
    ).convert("RGB")

    image = view_transform(image)

    image = image.unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        logits = view_classifier(image)

        pred = torch.argmax(
            logits,
            dim=1
        ).item()

    # 0 = AP/PA
    # 1 = LATERAL

    if pred == 0:
        return "AP"

    return "LATERAL"

In [13]:
@dataclass
class RetrievalResult:

    reports: List[Dict]

    fused_reports: List[Dict]

    windows: List[Dict]

    trajectories: List[Dict]

In [14]:
# ============================================================
# ProcessedQuery Structure
# ============================================================

@dataclass
class ProcessedQuery:

    report_embedding: np.ndarray

    fused_query: np.ndarray

    window_query: np.ndarray

    trajectory_query: np.ndarray

    temporal_status: str

    pathology_labels: List[str]

In [15]:
# Temporal Reranking
def rerank_temporal_alignment(
    temporal_intent,
    retrieved,
):

    reranked = []

    for item in retrieved:

        score = item["score"]

        trend = item.get("trend", "unknown")

        if trend == temporal_intent:
            score += 0.15

        reranked.append((score, item))

    reranked = sorted(
        reranked,
        key=lambda x: x[0],
        reverse=True,
    )

    return [x[1] for x in reranked]

In [16]:
# ============================================================
# Lightweight QueryProcessor
# ============================================================

class QueryProcessor:

    def __init__(self):

        self.pathology_keywords = {

            "edema": [
                "edema",
                "fluid overload",
            ],

            "effusion": [
                "pleural effusion",
                "effusion",
            ],

            "pneumonia": [
                "pneumonia",
                "consolidation",
                "opacity",
            ],

            "pneumothorax": [
                "pneumothorax",
            ],

            "cardiomegaly": [
                "cardiomegaly",
                "enlarged cardiac silhouette",
            ],
        }

    # ========================================================
    # PATHOLOGY EXTRACTION
    # ========================================================

    def extract_pathologies(
        self,
        text,
    ):

        text = text.lower()

        labels = []

        for pathology, keywords in self.pathology_keywords.items():

            if any(k in text for k in keywords):

                labels.append(pathology)

        return labels

    # ========================================================
    # TEMPORAL FEATURES
    # ========================================================

    def build_window_features(
        self,
        temporal_status,
    ):

        trend = 0.0

        if temporal_status == "improving":
            trend = -1.0

        elif temporal_status == "worsening":
            trend = 1.0

        intensity = abs(trend)

        return np.array(
            [trend, intensity],
            dtype=np.float32,
        )

    def build_trajectory_features(
        self,
        temporal_status,
        labels,
    ):

        improving = 0.0
        worsening = 0.0

        if temporal_status == "improving":
            improving = 1.0

        elif temporal_status == "worsening":
            worsening = 1.0

        severity = min(
            len(labels) / 5.0,
            1.0,
        )

        return np.array([
            improving,
            worsening,
            severity,
            float("edema" in labels),
            float("effusion" in labels),
            float("pneumonia" in labels),
            float("cardiomegaly" in labels),
            float("pneumothorax" in labels),
        ], dtype=np.float32)

    # ========================================================
    # MAIN PROCESS
    # ========================================================

    def process(
        self,
        findings_text,
        image_paths=None,
    ):

        # ====================================================
        # TEXT EMBEDDING
        # ====================================================

        text_emb = encode_text_query(
            findings_text
        )

        # ====================================================
        # IMAGE EMBEDDING
        # ====================================================

        image_emb = None

        if image_paths:

            img_embs = []

            for p in image_paths:

                emb = encode_image_query(p)

                img_embs.append(emb)

            image_emb = np.mean(
                img_embs,
                axis=0,
            )

        # ====================================================
        # FUSED QUERY
        # ====================================================

        fused_query = text_emb.copy()

        if image_emb is not None:

            fused_query = (
                0.7 * text_emb +
                0.3 * image_emb
            )

        fused_query = fused_query.astype(
            np.float32
        )

        # ====================================================
        # TEMPORAL STATUS
        # ====================================================

        temporal_status = detect_temporal_intent(
            findings_text
        )

        # ====================================================
        # PATHOLOGY LABELS
        # ====================================================

        pathology_labels = self.extract_pathologies(
            findings_text
        )

        # ====================================================
        # WINDOW QUERY
        # ====================================================

        window_features = self.build_window_features(
            temporal_status
        )

        window_query = np.concatenate([
            fused_query,
            window_features,
        ])

        # ====================================================
        # TRAJECTORY QUERY
        # ====================================================

        trajectory_features = self.build_trajectory_features(
            temporal_status,
            pathology_labels,
        )

        trajectory_query = np.concatenate([
            fused_query,
            trajectory_features,
        ]).astype(np.float32)

        return ProcessedQuery(

            report_embedding=text_emb,

            fused_query=fused_query,

            window_query=window_query,

            trajectory_query=trajectory_query,

            temporal_status=temporal_status,

            pathology_labels=pathology_labels,
        )

In [17]:
class TemporalRetriever:

    def __init__(self):
        pass

    def retrieve(
        self,
        query,
        top_k=5,
    ):

        temporal_intent = query.temporal_status

        report_query = query.report_embedding

        fused_query = query.fused_query
        
        window_query = query.window_query
        
        trajectory_query = query.trajectory_query

        report_query = report_query.reshape(1, -1)
        fused_query = fused_query.reshape(1, -1)
        window_query = window_query.reshape(1, -1)
        trajectory_query = trajectory_query.reshape(1, -1)
        
        faiss.normalize_L2(report_query)
        faiss.normalize_L2(fused_query)
        faiss.normalize_L2(window_query)
        faiss.normalize_L2(trajectory_query)

        # ==========================================
        # REPORT RETRIEVAL
        # ==========================================

        scores, indices = report_index.search(
            report_query,
            top_k,
        )

        reports = []

        for score, idx in zip(scores[0], indices[0]):

            study_key = report_keys[idx]

            report = parsed_reports.get(
                study_key,
                {}
            )

            reports.append({
                "study_key": study_key,
                "score": float(score),
                "findings": report.get("findings_text", ""),
                "impression": report.get("impression_text", ""),
            })

        # ==========================================
        # FUSED RETRIEVAL
        # ==========================================
        
        scores, indices = fused_index.search(
            fused_query,
            top_k,
        )
        
        fused_reports = []
        
        for score, idx in zip(scores[0], indices[0]):
        
            study_key = report_keys[idx]
        
            report = parsed_reports.get(
                study_key,
                {}
            )
        
            fused_reports.append({
                "study_key": study_key,
                "score": float(score),
                "findings": report.get("findings_text", ""),
                "impression": report.get("impression_text", ""),
            })

        # ==========================================
        # WINDOW RETRIEVAL
        # ==========================================

        scores, indices = window_index.search(
            window_query,
            top_k,
        )

        windows = []

        for score, idx in zip(scores[0], indices[0]):

            window_key = window_keys[idx]
            
            meta = window_metadata.get(
                window_key,
                {}
            )

            windows.append({
                "window_key": meta.get(
                    "window_key",
                    window_key
                ),
                "score": float(score),
                "trend": meta.get("trend", "unknown"),
                "text": meta.get("text", ""),
            })

        windows = rerank_temporal_alignment(
            temporal_intent,
            windows,
        )

        # ==========================================
        # TRAJECTORY RETRIEVAL
        # ==========================================

        scores, indices = trajectory_index.search(
            trajectory_query,
            top_k,
        )

        trajs = []

        for score, idx in zip(scores[0], indices[0]):

            traj_key = trajectory_keys[idx]
            
            meta = trajectory_metadata.get(
                traj_key,
                {}
            )

            trajs.append({
                "patient_id": meta.get(
                    "patient_id",
                    traj_key
                ),
                "score": float(score),
                "trend": meta.get("trend", "unknown"),
                "summary": meta.get("summary", ""),
            })

        trajs = rerank_temporal_alignment(
            temporal_intent,
            trajs,
        )

        return RetrievalResult(
            reports=reports,
            fused_reports=fused_reports,
            windows=windows,
            trajectories=trajs,
        )

In [18]:
# ============================================================
# TEMPORAL MULTIMODAL RAG PROMPT
# ============================================================

def build_prompt(

    patient_timeline_text,
    retrieval_result,
):

    # ========================================================
    # RETRIEVED REPORTS
    # ========================================================

    reports_text = "\n".join([

        r["findings"][:200]

        for r in retrieval_result.reports[:3]
    ])


    # ========================================================
    # MULTIMODAL RETRIEVAL
    # ========================================================

    fused_text = "\n".join([

        r["findings"][:200]

        for r in retrieval_result.fused_reports[:3]
    ])


    # ========================================================
    # TEMPORAL WINDOWS
    # ========================================================

    windows_text = "\n".join([

        w["text"][:200]

        for w in retrieval_result.windows[:3]
    ])


    # ========================================================
    # TRAJECTORIES
    # ========================================================

    trajectory_text = "\n".join([

        t["summary"][:200]

        for t in retrieval_result.trajectories[:3]
    ])


    # ========================================================
    # FINAL PROMPT
    # ========================================================

    prompt = f"""
You are an expert thoracic radiologist AI specialized in
longitudinal chest X-ray interpretation.

PATIENT TIMELINE CONTEXT:
{patient_timeline_text}

RETRIEVED PRIOR REPORTS:
{reports_text}

RETRIEVED MULTIMODAL CASES:
{fused_text}

TEMPORAL PROGRESSION MEMORY:
{windows_text}

LONGITUDINAL TRAJECTORY MEMORY:
{trajectory_text}

TASK:
Generate a clinically grounded chest X-ray report
for the MOST RECENT study using temporal reasoning
across the provided timeline and retrieved memories.

Focus on:
- interval progression
- worsening vs improvement
- persistent findings
- disease evolution
- temporal consistency

Output ONLY:

FINDINGS:
...

IMPRESSION:
...

Rules:
- Use concise radiology-report style
- No bullet points
- No conversational text
- No explanations
- Keep longitudinal reasoning clinically grounded
"""

    return prompt.strip()


def retrieval_baseline(retrieval_result):

    findings = []
    impressions = []

    for r in retrieval_result.reports[:3]:

        findings.append(r.get("findings", ""))
        impressions.append(r.get("impression", ""))

    findings = " ".join(findings)
    impressions = " ".join(impressions)

    return f"""
FINDINGS:
{findings}

IMPRESSION:
{impressions}
"""

In [19]:
# ============================================================
# Evaluation Metrics
# ============================================================

PATHOLOGY_TERMS = {
    "Atelectasis": ["atelectasis", "collapse"],
    "Cardiomegaly": ["cardiomegaly", "heart enlarged"],
    "Consolidation": ["consolidation"],
    "Edema": ["edema", "congestion"],
    "Enlarged Cardiomediastinum": ["mediastinum"],
    "Fracture": ["fracture"],
    "Lung Lesion": ["nodule", "mass", "lesion"],
    "Lung Opacity": ["opacity", "infiltrate", "ground glass"],
    "No Finding": ["normal", "clear"],
    "Pleural Effusion": ["effusion"],
    "Pleural Other": ["pleural thickening"],
    "Pneumonia": ["pneumonia"],
    "Pneumothorax": ["pneumothorax"],
    "Support Devices": ["tube", "line", "catheter", "device"],
}


# ============================================================
# TEXT CLEANING
# ============================================================

def clean_text(text):

    text = text.lower()

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()

# Sentence splitting for pathology extraction context windows
def split_sentences(text):

    text = clean_text(text)

    sentences = re.split(
        r"[.\n;]+",
        text,
    )

    return [

        s.strip()

        for s in sentences

        if len(s.strip()) > 0
    ]


# ============================================================
# NEGATION-AWARE PATHOLOGY EXTRACTION
# ============================================================

NEGATION_WORDS = [
    "no",
    "without",
    "absent",
    "negative for",
]


def extract_labels(text):

    text = clean_text(text)

    labels = {}


    # ========================================================
    # EACH PATHOLOGY GROUP
    # ========================================================

    for pathology, keywords in PATHOLOGY_TERMS.items():

        found = False

        negated = False


        # ====================================================
        # KEYWORD SEARCH
        # ====================================================

        for kw in keywords:

            matches = list(
                re.finditer(
                    re.escape(kw.lower()),
                    text
                )
            )

            if not matches:
                continue


            found = True


            # =================================================
            # NEGATION WINDOW
            # =================================================

            idx = matches[-1].start()

            window = text[
                max(0, idx - 40): idx
            ]

            negated = any(

                neg_word in window

                for neg_word in NEGATION_WORDS
            )

            if negated:
                break


        # ====================================================
        # STORE LABEL
        # ====================================================

        if found:

            labels[pathology] = (

                "negative"

                if negated

                else "positive"
            )

    return labels


# ============================================================
# BLEU
# ============================================================

from nltk.translate.bleu_score import (
    sentence_bleu,
    SmoothingFunction,
)

smoothie = SmoothingFunction().method4


def compute_bleu(
    pred,
    gt,
):

    pred_tokens = clean_text(pred).split()

    gt_tokens = [
        clean_text(gt).split()
    ]

    try:

        score = sentence_bleu(

            gt_tokens,

            pred_tokens,

            smoothing_function=smoothie,
        )

    except:

        score = 0.0

    return score


# ============================================================
# ROUGE-L
# ============================================================

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True,
)

def compute_rouge(
    pred,
    gt,
):

    try:

        score = rouge.score(
            gt,
            pred,
        )["rougeL"].fmeasure

    except:

        score = 0.0

    return score


# ============================================================
# CHEXPERT F1
# ============================================================

def compute_chexpert_f1(
    pred,
    gt,
):

    pred_labels = extract_labels(pred)

    gt_labels = extract_labels(gt)

    all_keys = set(
        pred_labels.keys()
    ) | set(
        gt_labels.keys()
    )

    if len(all_keys) == 0:

        return 1.0

    tp = 0
    fp = 0
    fn = 0

    for key in all_keys:

        pred_val = pred_labels.get(key)
        gt_val = gt_labels.get(key)

        if pred_val == gt_val and pred_val is not None:

            tp += 1

        elif pred_val is not None and gt_val is None:

            fp += 1

        elif pred_val != gt_val:

            fp += 1
            fn += 1

    precision = tp / (tp + fp + 1e-8)

    recall = tp / (tp + fn + 1e-8)

    f1 = (
        2 * precision * recall /
        (precision + recall + 1e-8)
    )

    return f1

# ─────────────────────────────────────────────
# TEMPORAL CHANGE PATTERNS
# ─────────────────────────────────────────────

IMPROVE = [r"improv", r"resolv", r"decreas", r"reduced", r"less", r"clearing"]
WORSEN  = [r"worsen", r"increas", r"progress", r"larger", r"worse"]
STABLE  = [r"unchanged", r"stable", r"no\s+change", r"similar", r"persistent"]
NEW     = [r"new", r"developed", r"appeared", r"interval\s+development"]
RESOLVED= [r"resolved", r"no\s+longer", r"disappeared", r"cleared"]

# ============================================================
# TEMPORAL PATTERN GROUPS
# ============================================================

TEMPORAL_PATTERNS = {

    "improved": IMPROVE,

    "worsened": WORSEN,

    "stable": STABLE,

    "new": NEW,

    "resolved": RESOLVED,
}


def match_pathologies_in_sentence(sentence):

    found = []

    for pathology, keywords in PATHOLOGY_TERMS.items():

        for kw in keywords:

            if kw.lower() in sentence:

                found.append(pathology)

                break

    return found

# Sentence level detector
def detect_sentence_trend(sentence):

    for state, patterns in TEMPORAL_PATTERNS.items():

        for pattern in patterns:

            if re.search(pattern, sentence):

                return state

    return "present"

# ============================================================
# TEMPORAL STATE EXTRACTION
# ============================================================

def extract_temporal_states(text):

    text = clean_text(text)

    states = set()

    for state, patterns in TEMPORAL_PATTERNS.items():

        for pattern in patterns:

            if re.search(pattern, text):

                states.add(state)

                break

    if len(states) == 0:

        states.add("unknown")

    return states


# ============================================================
# TEMPORAL CHANGE EXTRACTION
# ============================================================

def extract_temporal_changes(text):

    text = clean_text(text)

    changes = set()

    for state, patterns in TEMPORAL_PATTERNS.items():

        for pattern in patterns:

            if re.search(pattern, text):

                changes.add(state)

                break

    return changes


# ============================================================
# TEMPORAL AGREEMENT
# ============================================================

def compute_temporal_agreement(
    pred,
    gt,
):

    pred_states = extract_temporal_states(pred)

    gt_states = extract_temporal_states(gt)

    for state in gt_states:

        if state == "unknown":
            continue

        change_timestamp_stats[
            state
        ].setdefault(
            timestamp,
            []
        )

        change_timestamp_stats[
            state
        ][timestamp].append(

            int(
                state
                in
                pred_states
            )
        )

    if "unknown" in gt_states:

        return 1.0

    intersection = len(
        pred_states & gt_states
    )

    union = len(
        pred_states | gt_states
    )

    return intersection / (union + 1e-8)


# ============================================================
# PATHOLOGY-SPECIFIC TEMPORAL EXTRACTION
# ============================================================

def extract_pathology_progression(text):

    sentences = split_sentences(text)

    progression = {}

    for sentence in sentences:

        pathologies = match_pathologies_in_sentence(
            sentence
        )

        if len(pathologies) == 0:
            continue

        trend = detect_sentence_trend(
            sentence
        )

        for pathology in pathologies:

            progression[pathology] = trend

    return progression


def _pathology_keys(text):

    return set(extract_pathology_progression(text).keys())


def compute_pathology_progression_accuracy(
    pred,
    gt,
):

    pred_prog = extract_pathology_progression(
        pred
    )

    gt_prog = extract_pathology_progression(
        gt
    )

    if len(pred_prog) == 0 and len(gt_prog) == 0:

        return 1.0

    if len(pred_prog) == 0 or len(gt_prog) == 0:

        return 0.0

    all_pathologies = set(
        pred_prog.keys()
    ) | set(
        gt_prog.keys()
    )

    correct = 0

    for pathology in all_pathologies:

        if pred_prog.get(pathology) == gt_prog.get(pathology):

            correct += 1

    return correct / len(all_pathologies)


def extract_pathology_temporal_states(text):

    text = clean_text(text)

    result = {}

    sentences = re.split(r"[.\n]+", text)

    for sentence in sentences:

        sentence = sentence.strip()

        if len(sentence) == 0:
            continue

        detected_state = None

        for state, patterns in TEMPORAL_PATTERNS.items():

            if any(
                re.search(p, sentence)
                for p in patterns
            ):
                detected_state = state
                break

        if detected_state is None:
            continue

        for pathology, keywords in PATHOLOGY_TERMS.items():

            if any(
                kw.lower() in sentence
                for kw in keywords
            ):

                result[pathology] = detected_state

    return result
    

# ============================================================
# LONGITUDINAL ENTITY CONSISTENCY
# ============================================================

def compute_longitudinal_consistency(

    generated_reports,
):

    if len(generated_reports) < 2:

        return 1.0

    consistency_scores = []

    prev_labels = extract_labels(
        generated_reports[0]
    )

    for report in generated_reports[1:]:

        curr_labels = extract_labels(report)

        overlap = len(

            set(prev_labels.items()) &
            set(curr_labels.items())
        )

        total = len(

            set(prev_labels.items()) |
            set(curr_labels.items())
        )

        if total == 0:

            consistency_scores.append(1.0)

        else:

            consistency_scores.append(
                overlap / total
            )

        prev_labels = curr_labels

    return np.mean(consistency_scores)


# ============================================================
# RETRIEVAL PRECISION@K
# ============================================================

def compute_retrieval_precision(

    retrieved_reports,

    gt_text,
):

    gt_keys = _pathology_keys(gt_text)

    if len(gt_keys) == 0:

        return 1.0

    scores = []

    for r in retrieved_reports:

        retrieved_keys = _pathology_keys(
            r.get("findings", "")
        )

        if len(retrieved_keys) == 0:
            continue

        overlap = len(retrieved_keys & gt_keys)

        precision = overlap / len(gt_keys)

        scores.append(precision)

    if len(scores) == 0:

        return 0.0

    return np.mean(scores)


# ============================================================
# RETRIEVAL HIT@K
# ============================================================

def compute_retrieval_hit(

    retrieved_reports,

    gt_text,
):

    gt_keys = _pathology_keys(gt_text)

    if len(gt_keys) == 0:

        return 1.0

    for r in retrieved_reports:

        retrieved_keys = _pathology_keys(
            r.get("findings", "")
        )

        if len(retrieved_keys & gt_keys) > 0:

            return 1.0

    return 0.0

In [20]:
# VLM Backend Base
class VLMBackend:

    def generate(
        self,
        prompt,
        image_paths=None,
    ):
        raise NotImplementedError

In [21]:
class QwenBackend(VLMBackend):
    
    def __init__(
        self,
        model_name,
    ):

        cleanup_cuda()

        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            llm_int8_enable_fp32_cpu_offload=True,
        )

        self.processor = AutoProcessor.from_pretrained(
            model_name,
            trust_remote_code=True,
        )

        base_model = (
            Qwen2_5_VLForConditionalGeneration
            .from_pretrained(
                model_name,
                quantization_config=quant_config,
                low_cpu_mem_usage=True,
                torch_dtype=torch.float16,
                trust_remote_code=True,
            )
        )

        base_model = base_model.to(DEVICE)

        self.model = base_model
        self.model.eval()

    def generate(
        self,
        prompt,
        image_paths=None,
    ):
    
        content = []
    
        # ========================================================
        # IMAGES
        # ========================================================
    
        if image_paths:
    
            for img in image_paths[:2]:
    
                content.append({
                    "type": "image",
                    "image": img,
                })
    
        # ========================================================
        # TEXT
        # ========================================================
    
        content.append({
            "type": "text",
            "text": prompt,
        })
    
        messages = [
            {
                "role": "user",
                "content": content,
            }
        ]
    
        # ========================================================
        # CHAT TEMPLATE
        # ========================================================
    
        text_prompt = self.processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    
        # ========================================================
        # PROCESS INPUTS
        # ========================================================
    
        inputs = self.processor(
            text=[text_prompt],
            images=[
                Image.open(p).convert("RGB").resize((128, 128))
                for p in (image_paths[:2] if image_paths else [])
            ] if image_paths else None,
            return_tensors="pt",
            max_pixels=224*224,
        )
    
        model_device = next(self.model.parameters()).device
        inputs = {
            k: v.to(model_device)
            for k, v in inputs.items()
        }
    
        # ========================================================
        # GENERATION
        # ========================================================
    
        with torch.inference_mode():
    
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
            )
    
        # ========================================================
        # REMOVE PROMPT TOKENS
        # ========================================================
    
        generated_ids = outputs[
            :,
            inputs["input_ids"].shape[1]:
        ]
    
        # ========================================================
        # DECODE
        # ========================================================
    
        generated = self.processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
        )[0]

        del outputs
        del generated_ids
        del inputs
        
        cleanup_cuda()
    
        return generated.strip()

In [22]:
# class NVReasonCXRBackend:

#     def __init__(
#         self,
#         model_name,
#     ):

#         cleanup_cuda()

#         # ====================================================
#         # QUANTIZATION
#         # ====================================================

#         bnb_config = BitsAndBytesConfig(
#             load_in_4bit=True,
#             bnb_4bit_compute_dtype=torch.float16,
#             bnb_4bit_use_double_quant=True,
#             bnb_4bit_quant_type="nf4",
#             llm_int8_enable_fp32_cpu_offload=True,
#         )


#         # ====================================================
#         # PROCESSOR
#         # ====================================================

#         self.processor = AutoProcessor.from_pretrained(
#             model_name,
#             trust_remote_code=True,
#         )


#         # ====================================================
#         # MODEL
#         # ====================================================

#         base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#             model_name,
#             quantization_config=bnb_config,
#             low_cpu_mem_usage=True,
#             torch_dtype=torch.float16,
#             trust_remote_code=True,
#         )

#         base_model = base_model.to(DEVICE)

#         self.model = base_model
#         self.model.eval()

#     # ========================================================
#     # GENERATION
#     # ========================================================

#     def generate(
#         self,
#         prompt,
#         image_paths=None,
#     ):
    
#         content = []
    
#         # ========================================================
#         # IMAGES
#         # ========================================================
    
#         if image_paths:
    
#             for img in image_paths[:2]:
    
#                 content.append({
#                     "type": "image",
#                     "image": img,
#                 })
    
#         # ========================================================
#         # TEXT
#         # ========================================================
    
#         content.append({
#             "type": "text",
#             "text": prompt,
#         })
    
#         messages = [
#             {
#                 "role": "user",
#                 "content": content,
#             }
#         ]
    
#         # ========================================================
#         # CHAT TEMPLATE
#         # ========================================================
    
#         text_prompt = self.processor.apply_chat_template(
#             messages,
#             tokenize=False,
#             add_generation_prompt=True,
#         )
    
#         # ========================================================
#         # PROCESS INPUTS
#         # ========================================================
    
#         inputs = self.processor(
#             text=[text_prompt],
#             images=[
#                 Image.open(p).convert("RGB").resize((128, 128))
#                 for p in (image_paths[:2] if image_paths else [])
#             ] if image_paths else None,
#             return_tensors="pt",
#             max_pixels=224*224,
#         )
    
#         model_device = next(self.model.parameters()).device
#         inputs = {
#             k: v.to(model_device)
#             for k, v in inputs.items()
#         }
    
#         # ========================================================
#         # GENERATION
#         # ========================================================
    
#         with torch.inference_mode():
    
#             outputs = self.model.generate(
#                 **inputs,
#                 max_new_tokens=256,
#                 do_sample=False,
#             )
    
#         # ========================================================
#         # REMOVE PROMPT TOKENS
#         # ========================================================
    
#         generated_ids = outputs[
#             :,
#             inputs["input_ids"].shape[1]:
#         ]


#         # ========================================================
#         # DECODE
#         # ========================================================
    
#         generated = self.processor.batch_decode(
#             generated_ids,
#             skip_special_tokens=True,
#         )[0]

#         del outputs
#         del generated_ids
#         del inputs
        
#         cleanup_cuda()
    
#         return generated.strip()

In [23]:
# class MiniCPMBackend(VLMBackend):

#     def __init__(self, model_name):

#         cleanup_cuda()

#         bnb_config = BitsAndBytesConfig(
#             load_in_4bit=True,
#             bnb_4bit_compute_dtype=torch.float16,
#             bnb_4bit_quant_type="nf4",
#             bnb_4bit_use_double_quant=True,
#         )

#         self.processor = AutoProcessor.from_pretrained(
#             model_name,
#             trust_remote_code=True,
#         )

#         self.model = (
#             MiniCPMV4_6ForConditionalGeneration
#             .from_pretrained(
#                 model_name,
#                 quantization_config=bnb_config,
#                 torch_dtype=torch.float16,
#                 trust_remote_code=True,
#                 device_map="auto",
#             )
#         )

#         self.model.eval()

#     # ========================================================
#     # GENERATION
#     # ========================================================

#     def generate(
#         self,
#         prompt,
#         image_paths=None,
#     ):

#         content = []

#         # ====================================================
#         # IMAGES
#         # ====================================================

#         if image_paths:

#             for img in image_paths[:2]:

#                 content.append({

#                     "type": "image",

#                     "image": img,
#                 })

#         # ====================================================
#         # TEXT
#         # ====================================================

#         content.append({

#             "type": "text",

#             "text": prompt,
#         })

#         messages = [

#             {
#                 "role": "user",

#                 "content": content,
#             }
#         ]

#         # ====================================================
#         # CHAT TEMPLATE
#         # ====================================================

#         text_prompt = self.processor.apply_chat_template(

#             messages,

#             tokenize=False,

#             add_generation_prompt=True,
#         )

#         # ====================================================
#         # LOAD IMAGES
#         # ====================================================

#         images = [

#             Image.open(p)
#             .convert("RGB")
#             .resize((128,128))

#             for p in image_paths[:2]

#         ] if image_paths else None

#         # ====================================================
#         # PROCESS INPUTS
#         # ====================================================

#         inputs = self.processor(

#             text=[text_prompt],

#             images=images,

#             return_tensors="pt",
#         )

#         model_device = next(
#             self.model.parameters()
#         ).device

#         inputs = {

#             k: v.to(model_device)

#             for k, v in inputs.items()
#         }

#         # ====================================================
#         # GENERATE
#         # ====================================================

#         with torch.inference_mode():

#             outputs = self.model.generate(

#                 **inputs,

#                 max_new_tokens=256,

#                 do_sample=False,
#             )

#         # ====================================================
#         # REMOVE PROMPT
#         # ====================================================

#         generated_ids = outputs[
#             :,
#             inputs["input_ids"].shape[1]:
#         ]

#         # ====================================================
#         # DECODE
#         # ====================================================

#         generated = self.processor.batch_decode(

#             generated_ids,

#             skip_special_tokens=True,
#         )[0]

#         cleanup_cuda()

#         return generated.strip()


In [24]:
# ============================================================
# OUTPUT DIRECTORY
# ============================================================

OUTPUT_ROOT = Path("F:/Mimic")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# INITIALIZATION
# ============================================================

retriever = TemporalRetriever()
query_processor = QueryProcessor()

# ============================================================
# PATIENT-WISE TEMPORAL DATASET
# ============================================================

TEST_REPORT_DIR = Path(
    "F:/Mimic\mimic-cxr-p10-max9/reports"
)

TEST_IMAGE_DIR = Path(
    "F:/Mimic\mimic-cxr-p10-max9/images"
)

# ============================================================
# LOAD MIMIC METADATA
# ============================================================

metadata_df = pd.read_csv(

    "F:/Mimic/mimic-cxr-2.0.0-metadata.csv"
)

# ============================================================
# CREATE CHRONOLOGICAL DATETIME
# ============================================================

metadata_df["StudyDate"] = (
    metadata_df["StudyDate"]
    .astype(str)
)

metadata_df["StudyTime"] = (
    metadata_df["StudyTime"]
    .fillna(0)
    .astype(int)
    .astype(str)
    .str.zfill(6)
)

metadata_df["study_datetime"] = pd.to_datetime(

    metadata_df["StudyDate"]
    +
    metadata_df["StudyTime"],

    format="%Y%m%d%H%M%S",

    errors="coerce",
)

patient_trajectories = {}

print("Indexing image files...")

image_lookup = {}

for ext in ["jpg", "jpeg", "png"]:

    for img_path in TEST_IMAGE_DIR.rglob(f"*.{ext}"):

        try:

            # study folder name
            study_key = img_path.parent.name

            image_lookup.setdefault(
                study_key,
                []
            ).append(str(img_path))

        except:
            pass

print(
    f"Indexed {len(image_lookup)} study image groups"
)

print("Building patient trajectories...")

# ============================================================
# SCAN REPORTS
# ============================================================

for report_path in TEST_REPORT_DIR.rglob("*.txt"):

    try:

        # ====================================================
        # PATIENT ID
        # ====================================================

        report_rel_path = report_path.relative_to(TEST_REPORT_DIR)

        patient_id = report_rel_path.parts[0]

        # ====================================================
        # STUDY ID
        # ====================================================

        study_key = report_path.stem

        # ====================================================
        # REPORT
        # ====================================================

        with open(report_path) as f:

            report_text = f.read()

        # ====================================================
        # IMAGES
        # ====================================================

        image_paths = image_lookup.get(
            study_key,
            []
        )

        sample = {

            "study_key": study_key,

            "findings": report_text,

            "ground_truth": report_text,

            "image_paths": image_paths,
        }

        patient_trajectories.setdefault(
            patient_id,
            []
        ).append(sample)

    except Exception as e:

        print(f"Skipping {report_path}: {e}")

# ============================================================
# STUDY DATETIME LOOKUP
# ============================================================

study_datetime_lookup = {

    f"s{row.study_id}": row.study_datetime

    for _, row in metadata_df.iterrows()
}

# ============================================================
# SORT CHRONOLOGICALLY
# ============================================================

for patient_id in patient_trajectories:

    patient_trajectories[
        patient_id
    ] = sorted(

        patient_trajectories[
            patient_id
        ],

        key=lambda x:

        study_datetime_lookup.get(
    
            x["study_key"],
    
            pd.Timestamp.min
        )
    )

# ============================================================
# PATIENT LIST
# ============================================================

print(
    f"Built {len(patient_trajectories)} patient trajectories before sampling"
)

patient_ids = list(
    patient_trajectories.keys()
)

if MAX_TEST_SAMPLES is not None:

    patient_ids = patient_ids[
        :MAX_TEST_SAMPLES
    ]

print(
    f"Loaded {len(patient_ids)} patient trajectories"
)
print(f"Selected patient IDs: {patient_ids}")

# ============================================================
# MODEL INITIALIZATION
# ============================================================

if ACTIVE_MODEL == "qwen2.5-vl-3b":

    model = QwenBackend(
        VLM_MODELS["qwen2.5-vl-3b"]
    )

elif ACTIVE_MODEL == "nv-reason-cxr-3b":

    model = NVReasonCXRBackend(
        VLM_MODELS["nv-reason-cxr-3b"]
    )

elif ACTIVE_MODEL == "minicpm-v-4.6":

    model = MiniCPMBackend(
        VLM_MODELS["minicpm-v-4.6"]
    )

else:

    raise ValueError(
        f"Unknown model: {ACTIVE_MODEL}"
    )


from collections import defaultdict

disease_trajectory_stats = defaultdict(
    lambda: defaultdict(
        lambda: defaultdict(int)
    )
)

# ============================================================
# GENERATION LOOP
# ============================================================

all_outputs = []

save_path = (
    OUTPUT_ROOT /
    f"{ACTIVE_MODEL}_outputs_not_ft.jsonl"
)

with open(save_path, "w") as f:
    pass

for patient_idx, patient_id in enumerate(patient_ids):

    trajectory = patient_trajectories[
        patient_id
    ]

    print()
    print(
        f"[{patient_idx+1}/{len(patient_ids)}] "
        f"{patient_id}"
    )

    # ========================================================
    # TEMPORAL STUDIES
    # ========================================================

    for t in range(len(trajectory)):

        # ====================================================
        # CURRENT STUDY
        # ====================================================

        current = trajectory[t]

        # ====================================================
        # TEMPORAL POSITION
        # ====================================================

        timeline_position = t

        trajectory_length = len(
            trajectory
        )

        # ====================================================
        # PREVIOUS HISTORY
        # ====================================================

        history = trajectory[:t]

        # MAX_HISTORY = 5

        # history = trajectory[
        #     max(0, t-MAX_HISTORY):t
        # ]

        # ====================================================
        # BUILD TEMPORAL CONTEXT
        # ====================================================

        history_text = "\n".join([

            h["findings"]

            for h in history
        ])

        # ====================================================
        # CURRENT QUERY
        # ====================================================

        findings = current["findings"]

        gt = current["ground_truth"]

        image_paths = current["image_paths"]

        if len(history_text.strip()) > 0:
        
            temporal_query = f"""
        Previous studies:
        {history_text}
        
        Current study:
        {findings}
        """
        
        else:
        
            temporal_query = f"""
        Current study:
        {findings}
        """

        # ====================================================
        # QUERY PROCESSING
        # ====================================================

        processed_query = query_processor.process(
            findings_text=temporal_query,
            image_paths=image_paths,
        )

        retrieval = retriever.retrieve(
            processed_query
        )

        # ====================================================
        # PROMPT
        # ====================================================

        prompt = build_prompt(
            temporal_query,
            retrieval,
        )

        # ====================================================
        # GENERATION
        # ====================================================

        start_time = time.time()

        generated = model.generate(
            prompt=prompt,
            image_paths=image_paths,
        )

        elapsed = time.time() - start_time

        # ====================================================
        # SAVE
        # ====================================================

        record = {

            "patient_id": patient_id,

            "study_key": current["study_key"],

            "history_length": len(history),

            "timeline_position": timeline_position,

            "trajectory_length": trajectory_length,

            "query": temporal_query,

            "generated_report": generated,

            "ground_truth": gt,

            "retrieved_reports": retrieval.reports,

            "retrieved_windows": retrieval.windows,

            "retrieved_trajectories": retrieval.trajectories,

            "inference_time": elapsed,
        }

        all_outputs.append(record)

        with open(save_path, "a") as f:
            f.write(json.dumps(record) + "\n")

        cleanup_cuda()

print(f"Saved: {save_path.name}")

Indexing image files...
Indexed 1499 study image groups
Building patient trajectories...
Built 472 patient trajectories before sampling
Loaded 2 patient trajectories
Selected patient IDs: ['p10000032', 'p10000764']


Loading weights: 100%|██████████| 824/824 [00:12<00:00, 66.62it/s] 



[1/2] p10000032

[2/2] p10000764
Saved: qwen2.5-vl-3b_outputs_not_ft.jsonl


In [ ]:
# ============================================================
# HELPER: extract per-pathology temporal state from text
# (defined here to ensure availability regardless of cell order)
# ============================================================

def extract_pathology_temporal_states(text):

    text = clean_text(text)

    result = {}

    sentences = re.split(r"[.\n]+", text)

    for sentence in sentences:

        sentence = sentence.strip()

        if len(sentence) == 0:
            continue

        detected_state = None

        for state, patterns in TEMPORAL_PATTERNS.items():

            if any(
                re.search(p, sentence)
                for p in patterns
            ):
                detected_state = state
                break

        if detected_state is None:
            continue

        for pathology, keywords in PATHOLOGY_TERMS.items():

            if any(
                kw.lower() in sentence
                for kw in keywords
            ):

                result[pathology] = detected_state

    return result


# ============================================================
# FINAL EVALUATION
# ============================================================

outputs = all_outputs

bleu_scores = []

rouge_scores = []

f1_scores = []

temporal_scores = []

retrieval_scores = []

retrieval_hit_scores = []

pathology_progression_scores = []

temporal_by_timestamp = {}

pathology_progression_by_timestamp = {}

retrieval_hit_by_timestamp = {}

retrieval_precision_by_timestamp = {}

times = []

num_errors = 0

pathology_timestamp_stats = {
    pathology: {}
    for pathology in PATHOLOGY_TERMS.keys()
}

change_timestamp_stats = {

    "improved": {},

    "worsened": {},

    "stable": {},

    "new": {},

    "resolved": {},
}

from collections import defaultdict

disease_trajectory_stats = defaultdict(
    lambda: defaultdict(
        lambda: defaultdict(int)
    )
)

global_pathology_timeline = defaultdict(
    lambda: defaultdict(int)
)

# ============================================================
# GENERATED REPORTS
# ============================================================

generated_reports = []


# ============================================================
# SAMPLE LOOP
# ============================================================

for sample in outputs:

    timestamp = sample.get(
        "timeline_position",
        0
    )

    pred = sample.get(
        "generated_report",
        ""
    )

    gt = sample.get(
        "ground_truth",
        ""
    )

    if len(pred.strip()) == 0:

        num_errors += 1

        continue


    # ========================================================
    # STORE REPORT
    # ========================================================

    generated_reports.append(pred)


    # ========================================================
    # BLEU
    # ========================================================

    bleu_scores.append(
        compute_bleu(pred, gt)
    )


    # ========================================================
    # ROUGE
    # ========================================================

    rouge_scores.append(
        compute_rouge(pred, gt)
    )


    # ========================================================
    # CHEXPERT F1
    # ========================================================

    f1_scores.append(
        compute_chexpert_f1(pred, gt)
    )


    # ========================================================
    # TEMPORAL AGREEMENT
    # ========================================================

    temp_score = (
        compute_temporal_agreement(
            pred,
            gt,
        )
    )

    temporal_scores.append(
        temp_score
    )

    temporal_by_timestamp.setdefault(
        timestamp,
        []
    ).append(
        temp_score
    )


    # ========================================================
    # PATHOLOGY PROGRESSION ACCURACY
    # ========================================================
    pred_prog = extract_pathology_progression(
        pred
    )

    gt_prog = extract_pathology_progression(
        gt
    )

    gt_temporal = extract_pathology_temporal_states(gt)

    for pathology, state in gt_temporal.items():

        disease_trajectory_stats[
            pathology
        ][timestamp][state] += 1

        global_pathology_timeline[
            timestamp
        ][pathology] += 1

    prog_score = (
        compute_pathology_progression_accuracy(
            pred,
            gt
        )
    )

    pathology_progression_scores.append(
        prog_score
    )

    pathology_progression_by_timestamp.setdefault(
        timestamp,
        []
    ).append(
        prog_score
    )

    for pathology in sorted(
        set(pred_prog) | set(gt_prog)
    ):

        pathology_timestamp_stats[
            pathology
        ].setdefault(
            timestamp,
            []
        )

        pathology_timestamp_stats[
            pathology
        ][timestamp].append(

            int(
                pred_prog.get(pathology)
                ==
                gt_prog.get(pathology)
            )
        )

    
    # ========================================================
    # TEMPORAL CHANGE-TYPE ACCURACY
    # ========================================================
    gt_changes = extract_temporal_changes(
        gt
    )

    pred_changes = extract_temporal_changes(
        pred
    )

    for change_type in sorted(
        set(gt_changes) | set(pred_changes)
    ):

        if change_type not in change_timestamp_stats:
            continue

        change_timestamp_stats[
            change_type
        ].setdefault(
            timestamp,
            []
        )

        change_timestamp_stats[
            change_type
        ][timestamp].append(

            int(
                change_type
                in
                pred_changes
                and
                change_type
                in
                gt_changes
            )
        )

    # ========================================================
    # RETRIEVAL PRECISION
    # ========================================================

    retrieval_scores.append(

        compute_retrieval_precision(

            sample.get(
                "retrieved_reports",
                []
            ),

            gt,
        )
    )


    # ========================================================
    # RETRIEVAL HIT@K
    # ========================================================
    
    retrieval_hit_scores.append(
    
        compute_retrieval_hit(
    
            sample.get(
                "retrieved_reports",
                []
            ),
    
            gt,
        )
    )


    # ========================================================
    # INFERENCE TIME
    # ========================================================

    times.append(
        sample.get(
            "inference_time",
            0.0
        )
    )


# ============================================================
# LONGITUDINAL CONSISTENCY
# ============================================================

longitudinal_score = (
    compute_longitudinal_consistency(
        generated_reports
    )
)


# ============================================================
# RESULTS
# ============================================================

results = {

    "quantization": "4bit",

    "bleu":
    
    float(np.mean(
        bleu_scores
    ))
    
    if len(bleu_scores) > 0
    
    else 0.0,

    "rouge_l": 

    float(np.mean(
        rouge_scores
    ))
    
    if len(rouge_scores) > 0
    
    else 0.0,

    "chexpert_f1":
     
    float(np.mean(
        f1_scores
    ))
    
    if len(f1_scores) > 0
    
    else 0.0,

    "temporal_agreement":
        
    float(np.mean(
        temporal_scores
    ))
    
    if len(temporal_scores) > 0
    
    else 0.0,

    "pathology_progression_accuracy":
 
    float(np.mean(
        pathology_progression_scores
    ))
    
    if len(pathology_progression_scores) > 0
    
    else 0.0,

    "longitudinal_consistency":
    longitudinal_score,

    "retrieval_precision":
   
    float(np.mean(
        retrieval_scores
    ))
    
    if len(retrieval_scores) > 0
    
    else 0.0,

    "retrieval_hit@k":

    float(np.mean(
        retrieval_hit_scores
    ))
    
    if len(retrieval_hit_scores) > 0
    
    else 0.0,

    "num_patients": len(
        patient_ids
    ),

    "num_generated_reports":
    len(generated_reports),

    "valid_report_ratio":
    len(generated_reports) /
    (max(len(outputs), 1)),

    "num_errors":
    num_errors,

    "avg_time_per_sample_s":

    float(np.mean(
        times
    ))
    
    if len(times) > 0
    
    else 0.0,
}


# ============================================================
# PRINT
# ============================================================

print()

print(f"[{ACTIVE_MODEL}] Results:")

for k, v in results.items():

    if isinstance(v, float):

        print(f"    {k}: {v:.4f}")

    else:

        print(f"    {k}: {v}")

In [ ]:
for pathology in PATHOLOGY_TERMS:

    ts_data = disease_trajectory_stats[pathology]

    if len(ts_data) == 0:
        continue

    print(pathology)

    for ts in sorted(ts_data.keys()):

        print(
            ts,
            dict(ts_data[ts])
        )

In [ ]:
# ============================================================
# PRINT: How each disease state evolves across timestamps
# (shows actual counts/proportions, not accuracy)
# ============================================================

print("\n" + "="*60)
print("DISEASE PROGRESSION OVER TIME (Ground-Truth States)")
print("="*60)

ALL_STATES = ["new", "worsened", "stable", "improved", "resolved", "present"]

for pathology in sorted(PATHOLOGY_TERMS.keys()):

    ts_data = disease_trajectory_stats.get(pathology, {})

    if len(ts_data) == 0:
        continue

    print(f"\n{pathology}")
    print("-" * len(pathology))

    for ts in sorted(ts_data.keys()):

        state_counts = dict(ts_data[ts])
        total = sum(state_counts.values())

        if total == 0:
            continue

        parts = []
        for state in ALL_STATES:
            cnt = state_counts.get(state, 0)
            if cnt > 0:
                pct = 100.0 * cnt / total
                parts.append(f"{state}: {cnt} ({pct:.0f}%)")

        print(f"  T{ts} (n={total}): " + ", ".join(parts))


In [ ]:
print("\nTEMPORAL AGREEMENT BY TIMESTAMP")

for ts in sorted(
    temporal_by_timestamp
):

    print(

        f"T{ts}: "

        f"{np.mean(temporal_by_timestamp[ts]):.4f}"

        f" (n={len(temporal_by_timestamp[ts])})"
    )

In [ ]:
print("\nPATHOLOGY PROGRESSION BY TIMESTAMP")

for ts in sorted(
    pathology_progression_by_timestamp
):

    print(

        f"T{ts}: "

        f"{np.mean(pathology_progression_by_timestamp[ts]):.4f}"
        f" (n={len(pathology_progression_by_timestamp[ts])})"
    )


print("\nPATHOLOGY PROGRESSION BY TIMESTAMP AND LABEL")

for pathology in sorted(pathology_timestamp_stats):

    # ==========================================
    # TOTAL SUPPORT
    # ==========================================

    total_occurrences = sum(

        len(values)

        for values in
        pathology_timestamp_stats[
            pathology
        ].values()
    )

    print(
        f"{pathology} "
        f"(total n={total_occurrences})"
    )

    for ts in sorted(
        pathology_timestamp_stats[
            pathology
        ]
    ):

        values = pathology_timestamp_stats[
            pathology
        ][ts]

        print(

            f"    T{ts}: "

            f"{np.mean(values):.4f} "

            f"(n={len(values)})"
        )


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns # Ensure you have seaborn installed (pip install seaborn)
from pathlib import Path

# Setup output directory
HEATMAP_DIR = OUTPUT_ROOT / "heatmaps"
HEATMAP_DIR.mkdir(exist_ok=True)

print("\nPATHOLOGY PROGRESSION HEATMAP...")

heatmap_data = []

# 1. Extract and flatten the data into a list of dictionaries
for pathology in sorted(pathology_timestamp_stats):
    
    # Calculate the total number of reports (n) for this specific pathology across all timestamps
    total_n = sum(len(values) for values in pathology_timestamp_stats[pathology].values())
    
    # Create the new label with the (n=X) format
    pathology_label = f"{pathology} (n={total_n})"

    for ts in sorted(pathology_timestamp_stats[pathology]):
        
        values = pathology_timestamp_stats[pathology][ts]
        
        if len(values) > 0:
            heatmap_data.append({
                "Pathology": pathology_label,  # Use the newly formatted label here
                "Timestamp": f"T{ts}",
                "Mean Label": np.mean(values)
            })

# Check if data exists before plotting
if not heatmap_data:
    print("No data available to plot.")
else:
    # 2. Convert to DataFrame and Pivot
    df = pd.DataFrame(heatmap_data)
    
    # Pivot transforms the long data into a 2D grid: 
    # Rows = Pathology, Columns = Timestamp, Cells = Mean Label
    pivot_df = df.pivot(
        index="Pathology", 
        columns="Timestamp", 
        values="Mean Label"
    )
    
    # 3. Plot the Heatmap
    plt.figure(figsize=(12, 8))
    
    sns.heatmap(
        pivot_df, 
        annot=True,         # Show the numerical values in the cells
        fmt=".4f",          # Match your original 4-decimal precision
        cmap="YlGnBu",      # Yellow-Green-Blue color palette (easily readable)
        cbar_kws={'label': 'Accuracy'},  # Label for the colorbar
        linewidths=0.5,     # Add gridlines between cells
    )
    
    plt.title("Pathology Progression Accuracy by Timestamp", pad=20, fontsize=14)
    plt.xlabel("Timestamp", fontsize=12)
    plt.ylabel("Pathology", fontsize=12)
    
    # Rotate the y-axis labels so they are horizontal and easy to read
    plt.yticks(rotation=0)
    
    plt.tight_layout()
    
    # Save the plot
    save_file = HEATMAP_DIR / "pathology_mean_progression_heatmap.png"
    plt.savefig(
        save_file, 
        dpi=300, 
        bbox_inches="tight"
    )
    
    plt.show()
    plt.close()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

STATES = [
    "new",
    "worsened",
    "stable",
    "improved",
    "resolved",
]

SAVE_DIR = OUTPUT_ROOT / "trajectory_fraction_plots"
SAVE_DIR.mkdir(exist_ok=True)

# Grab the pathologies and ensure we take up to 16 to fit the 4x4 grid
pathologies = sorted(disease_trajectory_stats.keys())[:16]

# Create a 4x4 subplot grid. Figure size is increased to (20, 20) for the extra row.
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(22, 22), sharey=True)

# Flatten the 2D array of axes into a 1D array so we can easily iterate over it
axes = axes.flatten()

for i, pathology in enumerate(pathologies):
    ax = axes[i]
    ts_data = disease_trajectory_stats[pathology]

    if len(ts_data) == 0:
        continue

    rows = []

    timestamps = sorted(
        ts_data.keys(),
        key=lambda x: int(str(x).replace("T", ""))
    )

    for ts in timestamps:
        total = sum(ts_data[ts].values())

        if total == 0:
            continue

        row = {"timestamp": ts}

        for state in STATES:
            row[state] = ts_data[ts].get(state, 0) / total

        rows.append(row)

    if len(rows) == 0:
        continue

    df = pd.DataFrame(rows)

    # Plot on the specific subplot using a scatter plot for each state
    for state in STATES:
        # Draw the scatter points
        ax.scatter(
            df["timestamp"],
            df[state],
            label=state,
            alpha=0.85,
            s=40, # Size of the scatter points
            zorder=3 # Ensures points are drawn on top of the lines
        )
        
        # Draw a faint connecting line to show the temporal path clearly
        ax.plot(
            df["timestamp"],
            df[state],
            alpha=0.3,
            linewidth=1.5,
            zorder=2
        )

    ax.set_title(f"{pathology}")
    
    # Rotate x-axis labels to prevent overlap in the smaller subplots
    ax.tick_params(axis='x', labelrotation=45)
    
    ax.set_xlabel("Timestamp")
    
    # Only show the y-label on the leftmost plots to reduce visual clutter
    if i % 4 == 0:
        ax.set_ylabel("Fraction of Reports")

    ax.set_ylim(0, 1)

# Hide any extra subplots if there are fewer than 16 pathologies in the dictionary
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Extract handles and labels for the legend from the last plotted axis
# Because we plot lines and points, Matplotlib might duplicate handles. 
# We grab only the first 'len(STATES)' handles to keep the legend clean.
handles, labels = ax.get_legend_handles_labels()
handles = handles[:len(STATES)]
labels = labels[:len(STATES)]

# Create a single legend at the top right, stacked vertically (top-to-down)
fig.legend(
    handles, 
    labels, 
    loc='upper right', 
    bbox_to_anchor=(1.05, 0.95), 
    ncol=1
)

fig.suptitle(
    "Temporal Evolution of Different Diseases",
    fontsize=20
)

plt.tight_layout()

# Save the single grid image
save_file = SAVE_DIR / "all_pathologies_grid_scatter.png"

plt.savefig(
    save_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

In [ ]:
rows = []

for ts in sorted(
    global_pathology_timeline.keys(),
    key=lambda x: int(
        str(x).replace("T","")
    )
):

    row = {
        "timestamp": ts
    }

    for pathology in PATHOLOGY_TERMS:

        row[pathology] = (
            global_pathology_timeline[
                ts
            ].get(
                pathology,
                0
            )
        )

    rows.append(row)

df_global = pd.DataFrame(rows)


plt.figure(figsize=(14,7))

plt.stackplot(

    df_global["timestamp"],

    *[
        df_global[p]

        for p in PATHOLOGY_TERMS
    ],

    labels=list(
        PATHOLOGY_TERMS.keys()
    ),

    alpha=0.85,
)

plt.title(
    "Global Pathology Burden Across Time"
)

plt.xlabel("Timestamp")

plt.ylabel(
    "Number of Studies"
)

plt.legend(
    bbox_to_anchor=(1.05,1),
    loc="upper left"
)

plt.tight_layout()

plt.savefig(

    OUTPUT_ROOT /
    "global_pathology_burden.png",

    dpi=300,

    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RADAR_DIR = OUTPUT_ROOT / "radar_plots"
RADAR_DIR.mkdir(exist_ok=True)

STATES = [
    "stable",
    "improved",
    "worsened",
    "new",
    "resolved",
]

# Get sorted timestamps
timestamps = sorted(
    {
        ts
        for pathology in disease_trajectory_stats
        for ts in disease_trajectory_stats[pathology]
    },
    key=lambda x: int(str(x).replace("T", ""))
)

# Limit to the first 9 timestamps to fit the 3x3 grid perfectly
timestamps_to_plot = timestamps[:9]

# 1. Create a 3x3 subplot grid with polar axes
fig, axes = plt.subplots(
    nrows=3, 
    ncols=3, 
    figsize=(20, 20), 
    subplot_kw=dict(polar=True) # CRITICAL: This makes all 9 subplots radar plots
)

# Flatten the axes array for easy iteration
axes = axes.flatten()

for i, timestamp in enumerate(timestamps_to_plot):
    
    ax = axes[i]
    diseases = []
    
    state_values = {state: [] for state in STATES}

    for pathology in PATHOLOGY_TERMS:
        diseases.append(pathology)
        
        ts_data = disease_trajectory_stats.get(pathology, {}).get(timestamp, {})
        total = sum(ts_data.values())

        for state in STATES:
            if total == 0:
                state_values[state].append(0)
            else:
                state_values[state].append(ts_data.get(state, 0) / total)

    N = len(diseases)
    
    # Calculate angles for the radar plot
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False)
    angles = np.concatenate((angles, [angles[0]]))

    ax.set_ylim(0, 1)

    for state in STATES:
        # Append the first value to the end to close the circular plot
        values = state_values[state] + [state_values[state][0]]

        # 2. Plot directly onto the specific axis (ax)
        ax.plot(
            angles,
            values,
            linewidth=2,
            label=state
        )

        ax.fill(
            angles,
            values,
            alpha=0.10
        )

    ax.set_xticks(angles[:-1])
    
    # Slightly smaller font size to prevent overlap in the grid
    ax.set_xticklabels(diseases, fontsize=12)

    ax.set_title(f"T{timestamp}", pad=25, fontsize=14)

# 3. Hide any extra subplots if there are fewer than 9 timestamps
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# 4. Extract handles and labels for a single unified legend
handles, labels = ax.get_legend_handles_labels()

# Create a single legend at the top right, stacked vertically (top-to-down)
fig.legend(
    handles, 
    labels, 
    loc="upper right", 
    bbox_to_anchor=(1.05, 0.95), 
    ncol=1, 
    fontsize=12,
    title="Progression States"
)

fig.suptitle(
    "Disease Progression States Across Timestamps",
    fontsize=20
)

# Adjust layout to make room for the suptitle at the top AND the legend on the right.
# We reduce the right boundary to 0.90 to give the vertical legend plenty of breathing room.
plt.tight_layout()

save_file = RADAR_DIR / "all_timestamps_radar_grid.png"

plt.savefig(
    save_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

In [ ]:
print("\nTEMPORAL CHANGE TYPE ACCURACY BY TIMESTAMP")

for change_type in sorted(
    change_timestamp_stats
):

    print(
        f"{change_type}:"
    )

    for ts in sorted(
        change_timestamp_stats[change_type]
    ):

        print(
            f"    T{ts}: "
            f"{np.mean(change_timestamp_stats[change_type][ts]):.4f}"
            f" (n={len(change_timestamp_stats[change_type][ts])})"
        )

## For Comparative study

In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

def pathology_positive_vector(text, pathology):
    labels = extract_labels(text)
    return 1 if labels.get(pathology) == "positive" else 0

mimic_per_pathology_rows = []

mimic_eval_outputs = outputs  # or all_outputs

all_y_true = []
all_y_pred = []

for pathology in PATHOLOGY_TERMS.keys():
    y_true = []
    y_pred = []

    for item in mimic_eval_outputs:
        gt = item.get("ground_truth", "")
        pred = item.get("generated_report", "")

        if len(pred.strip()) == 0:
            continue

        true_label = pathology_positive_vector(gt, pathology)
        pred_label = pathology_positive_vector(pred, pathology)

        y_true.append(true_label)
        y_pred.append(pred_label)

    all_y_true.extend(y_true)
    all_y_pred.extend(y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    mimic_per_pathology_rows.append({
        "Pathology": pathology,
        "Support_GT_Positive": int(sum(y_true)),
        "Predicted_Positive": int(sum(y_pred)),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
    })

mimic_per_pathology_df = pd.DataFrame(mimic_per_pathology_rows)

# Macro-average: simple mean across pathologies
macro_precision = mimic_per_pathology_df["Precision"].mean()
macro_recall = mimic_per_pathology_df["Recall"].mean()
macro_f1 = mimic_per_pathology_df["F1"].mean()

# Micro-average: aggregate all pathology decisions together
micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(
    all_y_true,
    all_y_pred,
    average="binary",
    zero_division=0
)

average_rows = pd.DataFrame([
    {
        "Pathology": "Macro-average",
        "Support_GT_Positive": int(mimic_per_pathology_df["Support_GT_Positive"].sum()),
        "Predicted_Positive": int(mimic_per_pathology_df["Predicted_Positive"].sum()),
        "TP": int(mimic_per_pathology_df["TP"].sum()),
        "FP": int(mimic_per_pathology_df["FP"].sum()),
        "FN": int(mimic_per_pathology_df["FN"].sum()),
        "TN": int(mimic_per_pathology_df["TN"].sum()),
        "Precision": float(macro_precision),
        "Recall": float(macro_recall),
        "F1": float(macro_f1),
    },
    {
        "Pathology": "Micro-average",
        "Support_GT_Positive": int(sum(all_y_true)),
        "Predicted_Positive": int(sum(all_y_pred)),
        "TP": int(mimic_per_pathology_df["TP"].sum()),
        "FP": int(mimic_per_pathology_df["FP"].sum()),
        "FN": int(mimic_per_pathology_df["FN"].sum()),
        "TN": int(mimic_per_pathology_df["TN"].sum()),
        "Precision": float(micro_precision),
        "Recall": float(micro_recall),
        "F1": float(micro_f1),
    },
])

mimic_per_pathology_df = mimic_per_pathology_df.sort_values(
    by=["Support_GT_Positive", "F1"],
    ascending=[False, False]
)

mimic_per_pathology_with_avg_df = pd.concat(
    [mimic_per_pathology_df, average_rows],
    ignore_index=True
)

display(mimic_per_pathology_with_avg_df)

mimic_per_pathology_with_avg_df.to_csv(
    "mimic_cxr_per_pathology_results_with_avg.csv",
    index=False
)

In [ ]:
# ============================================================
# IU CHEST X-RAY NON-TEMPORAL EVALUATION
# ============================================================

from pathlib import Path

import random
import re
import tarfile
import time
import xml.etree.ElementTree as ET

try:

    import numpy as np

except ImportError as exc:

    raise ImportError("numpy is required for the IU evaluation cell") from exc

missing_dependencies = [
    name
    for name in [
        "model",
        "cleanup_cuda",
        "compute_bleu",
        "compute_rouge",
        "compute_chexpert_f1",
    ]
    if name not in globals()
]

if missing_dependencies:

    raise RuntimeError(
        "Run the earlier notebook setup and model-loading cells before this IU evaluation cell. "
        f"Missing: {', '.join(missing_dependencies)}"
    )

IU_ROOT = Path("F:/Mimic/IU_ChestXray_Dataset")
IU_REPORTS_ARCHIVE = IU_ROOT / "NLMCXR_reports.tgz"
IU_IMAGES_ARCHIVE = IU_ROOT / "NLMCXR_png.tgz"
IU_EXTRACTED_IMAGE_DIR = IU_ROOT / "_extracted_images"
IU_MAX_SAMPLES = 100
IU_RANDOM_SEED = 42

print("Loading IU Chest X-ray samples...", flush=True)


def _normalize_text(text):

    return re.sub(r"\s+", " ", (text or "")).strip()


def _extract_xml_section(root, label):

    for abstract_text in root.findall(".//Abstract/AbstractText"):

        if (abstract_text.get("Label") or "").strip().upper() == label:

            return _normalize_text("".join(abstract_text.itertext()))

    return ""


def _extract_image_ids(root):

    image_ids = []

    for parent_image in root.findall(".//parentImage"):

        image_id = parent_image.get("id")

        if image_id:

            image_ids.append(image_id.strip())

    return image_ids


def _resolve_iu_member_name(member_names, image_id):

    candidate_names = [
        image_id,
        f"{image_id}.png",
        f"./{image_id}",
        f"./{image_id}.png",
    ]

    for candidate in candidate_names:

        normalized = candidate[2:] if candidate.startswith("./") else candidate

        if candidate in member_names:

            return candidate

        if normalized in member_names:

            return normalized

        if Path(normalized).name in member_names:

            return Path(normalized).name

    return None


def _extract_image_to_cache(image_tar, member_name):

    IU_EXTRACTED_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

    output_path = IU_EXTRACTED_IMAGE_DIR / Path(member_name).name

    if not output_path.exists():

        tar_member_name = image_member_names_by_key.get(member_name)

        if tar_member_name is None:

            candidate_names = [
                f"./{Path(member_name).name}",
                Path(member_name).name,
            ]

            for candidate in candidate_names:

                tar_member_name = image_member_names_by_key.get(candidate)

                if tar_member_name is not None:

                    break

        if tar_member_name is None:

            return None

        member = image_tar.getmember(tar_member_name)

        extracted = image_tar.extractfile(member)

        if extracted is None:

            return None

        with open(output_path, "wb") as output_file:

            output_file.write(extracted.read())

    return output_path


iu_samples = []
iu_missing_image_studies = 0

with tarfile.open(IU_REPORTS_ARCHIVE, "r:gz") as reports_tar, tarfile.open(IU_IMAGES_ARCHIVE, "r:gz") as images_tar:

    image_member_names_by_key = {}

    for image_index, member in enumerate(images_tar, start=1):

        if image_index == 1 or image_index % 1000 == 0:

            print(
                f"[IU load] Indexed {image_index} image members...",
                flush=True,
            )

        if not member.isfile():

            continue

        image_member_names_by_key[member.name] = member.name
        image_member_names_by_key[Path(member.name).name] = member.name

    report_members = []

    for report_index, member in enumerate(reports_tar, start=1):

        if report_index == 1 or report_index % 500 == 0:

            print(
                f"[IU load] Scanned {report_index} report members...",
                flush=True,
            )

        if member.isfile() and member.name.lower().endswith(".xml"):

            report_members.append(member)

    for report_index, report_member in enumerate(report_members, start=1):

        if report_index == 1 or report_index % 250 == 0:

            print(
                f"[IU load] Parsed {report_index}/{len(report_members)} reports...",
                flush=True,
            )

        report_file = reports_tar.extractfile(report_member)

        if report_file is None:

            continue

        root = ET.fromstring(report_file.read())

        findings = _extract_xml_section(root, "FINDINGS")
        impression = _extract_xml_section(root, "IMPRESSION")

        if not findings and not impression:

            continue

        image_member_names = []

        for image_id in _extract_image_ids(root):

            member_name = _resolve_iu_member_name(image_member_names_by_key, image_id)

            if member_name is None:

                continue

            image_member_names.append(member_name)

        image_member_names = list(dict.fromkeys(image_member_names))

        if not image_member_names:

            iu_missing_image_studies += 1
            continue

        ground_truth_parts = []

        if findings:

            ground_truth_parts.append(f"FINDINGS:\n{findings}")

        if impression:

            ground_truth_parts.append(f"IMPRESSION:\n{impression}")

        iu_samples.append(
            {
                "study_id": Path(report_member.name).stem,
                "image_member_names": image_member_names,
                "findings": findings,
                "ground_truth": "\n\n".join(ground_truth_parts),
            }
        )

random.Random(IU_RANDOM_SEED).shuffle(iu_samples)
iu_samples = iu_samples[:IU_MAX_SAMPLES]

with tarfile.open(IU_IMAGES_ARCHIVE, "r:gz") as images_tar:

    for sample in iu_samples:

        cached_image_paths = []

        for member_name in sample["image_member_names"]:

            cached_image_path = _extract_image_to_cache(images_tar, member_name)

            if cached_image_path is None or not cached_image_path.exists():

                continue

            cached_image_paths.append(str(cached_image_path))

        sample["image_paths"] = list(dict.fromkeys(cached_image_paths))

iu_samples = [sample for sample in iu_samples if sample["image_paths"]]

print(
    f"Resolved {len(iu_samples)} IU Chest X-ray samples "
    f"(skipped {iu_missing_image_studies} studies without resolvable images)"
)


def build_iu_prompt():

    return """You are an expert thoracic radiologist AI.

Generate a concise chest X-ray report for the provided image(s).

Output ONLY:

FINDINGS:
...

IMPRESSION:
...

Rules:
- Use concise radiology-report style
- No bullet points
- No conversational text
- No explanations
""".strip()


print("Running IU Chest X-ray generation and evaluation...")

iu_outputs = []
iu_bleu_scores = []
iu_rouge_scores = []
iu_f1_scores = []
iu_times = []
iu_num_errors = 0

for idx, sample in enumerate(iu_samples):

    print(f"[{idx + 1}/{len(iu_samples)}] {sample['study_id']}")

    prompt = build_iu_prompt()

    start_time = time.time()

    generated = model.generate(
        prompt=prompt,
        image_paths=sample["image_paths"],
    )

    elapsed = time.time() - start_time

    if len(generated.strip()) == 0:

        iu_num_errors += 1

        continue

    gt = sample["ground_truth"]

    iu_outputs.append(
        {
            "study_id": sample["study_id"],
            "image_paths": sample["image_paths"],
            "generated_report": generated,
            "ground_truth": gt,
            "inference_time": elapsed,
        }
    )

    iu_bleu_scores.append(
        compute_bleu(generated, gt)
    )

    iu_rouge_scores.append(
        compute_rouge(generated, gt)
    )

    iu_f1_scores.append(
        compute_chexpert_f1(generated, gt)
    )

    iu_times.append(elapsed)

    cleanup_cuda()


iu_results = {
    "dataset": "IU_ChestXray_Dataset",
    "num_samples": len(iu_samples),
    "num_generated_reports": len(iu_outputs),
    "num_errors": iu_num_errors,
    "bleu": float(np.mean(iu_bleu_scores)) if iu_bleu_scores else 0.0,
    "rouge_l": float(np.mean(iu_rouge_scores)) if iu_rouge_scores else 0.0,
    "chexpert_f1": float(np.mean(iu_f1_scores)) if iu_f1_scores else 0.0,
    "avg_time_per_sample_s": float(np.mean(iu_times)) if iu_times else 0.0,
}

print()
print("[IU Chest X-ray] Results:")
for key, value in iu_results.items():
    if isinstance(value, float):
        print(f"    {key}: {value:.4f}")
    else:
        print(f"    {key}: {value}")

In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

def pathology_positive_vector(text, pathology):
    labels = extract_labels(text)
    return 1 if labels.get(pathology) == "positive" else 0

mimic_per_pathology_rows = []

mimic_eval_outputs = iu_outputs  # or all_outputs

all_y_true = []
all_y_pred = []

for pathology in PATHOLOGY_TERMS.keys():
    y_true = []
    y_pred = []

    for item in mimic_eval_outputs:
        gt = item.get("ground_truth", "")
        pred = item.get("generated_report", "")

        if len(pred.strip()) == 0:
            continue

        true_label = pathology_positive_vector(gt, pathology)
        pred_label = pathology_positive_vector(pred, pathology)

        y_true.append(true_label)
        y_pred.append(pred_label)

    all_y_true.extend(y_true)
    all_y_pred.extend(y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    mimic_per_pathology_rows.append({
        "Pathology": pathology,
        "Support_GT_Positive": int(sum(y_true)),
        "Predicted_Positive": int(sum(y_pred)),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
    })

mimic_per_pathology_df = pd.DataFrame(mimic_per_pathology_rows)

# Macro-average: simple mean across pathologies
macro_precision = mimic_per_pathology_df["Precision"].mean()
macro_recall = mimic_per_pathology_df["Recall"].mean()
macro_f1 = mimic_per_pathology_df["F1"].mean()

# Micro-average: aggregate all pathology decisions together
micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(
    all_y_true,
    all_y_pred,
    average="binary",
    zero_division=0
)

average_rows = pd.DataFrame([
    {
        "Pathology": "Macro-average",
        "Support_GT_Positive": int(mimic_per_pathology_df["Support_GT_Positive"].sum()),
        "Predicted_Positive": int(mimic_per_pathology_df["Predicted_Positive"].sum()),
        "TP": int(mimic_per_pathology_df["TP"].sum()),
        "FP": int(mimic_per_pathology_df["FP"].sum()),
        "FN": int(mimic_per_pathology_df["FN"].sum()),
        "TN": int(mimic_per_pathology_df["TN"].sum()),
        "Precision": float(macro_precision),
        "Recall": float(macro_recall),
        "F1": float(macro_f1),
    },
    {
        "Pathology": "Micro-average",
        "Support_GT_Positive": int(sum(all_y_true)),
        "Predicted_Positive": int(sum(all_y_pred)),
        "TP": int(mimic_per_pathology_df["TP"].sum()),
        "FP": int(mimic_per_pathology_df["FP"].sum()),
        "FN": int(mimic_per_pathology_df["FN"].sum()),
        "TN": int(mimic_per_pathology_df["TN"].sum()),
        "Precision": float(micro_precision),
        "Recall": float(micro_recall),
        "F1": float(micro_f1),
    },
])

mimic_per_pathology_df = mimic_per_pathology_df.sort_values(
    by=["Support_GT_Positive", "F1"],
    ascending=[False, False]
)

mimic_per_pathology_with_avg_df = pd.concat(
    [mimic_per_pathology_df, average_rows],
    ignore_index=True
)

display(mimic_per_pathology_with_avg_df)

mimic_per_pathology_with_avg_df.to_csv(
    "iu_chestxray_per_pathology_results_with_avg.csv",
    index=False
)

## CheXpert Plus Evaluation

New cell added by Codex. Run this after the model setup and metric cells.


In [25]:
# ============================================================
# CHEXPERT PLUS NON-TEMPORAL EVALUATION
# 100 patient-level samples, analogous to IU Chest X-ray cell
# ============================================================

import json
import random
import re
import time
from pathlib import Path

try:
    import numpy as np
except ImportError as exc:
    raise ImportError("numpy is required for the CheXpert Plus evaluation cell") from exc

missing_dependencies = [
    name
    for name in [
        "model",
        "cleanup_cuda",
        "compute_bleu",
        "compute_rouge",
        "compute_chexpert_f1",
    ]
    if name not in globals()
]

if missing_dependencies:
    raise RuntimeError(
        "Run the earlier notebook setup and model-loading cells before this "
        "CheXpert Plus evaluation cell. "
        f"Missing: {', '.join(missing_dependencies)}"
    )

CHEXPERT_PLUS_ROOT = Path("F:/Mimic/CheXpert_Plus_Dataset")
CHEXPERT_PLUS_IMAGE_DIR = CHEXPERT_PLUS_ROOT / "images"
CHEXPERT_PLUS_LABEL_FILE = CHEXPERT_PLUS_ROOT / "labels" / "report_fixed.json"
CHEXPERT_PLUS_FINDINGS_RADGRAPH = (
    CHEXPERT_PLUS_ROOT / "RadGraph_Annotations" / "section_findings.json"
)
CHEXPERT_PLUS_IMPRESSION_RADGRAPH = (
    CHEXPERT_PLUS_ROOT / "RadGraph_Annotations" / "section_impression.json"
)

CHEXPERT_PLUS_MAX_PATIENTS = 100
CHEXPERT_PLUS_RANDOM_SEED = 42

print("Loading CheXpert Plus samples...", flush=True)


def _normalize_chexpert_text(text):
    return re.sub(r"\s+", " ", (text or "")).strip()


def _parse_chexpert_path(path_to_image):
    parts = Path(path_to_image).parts

    if parts and parts[0].lower() in {"train", "valid", "test"}:
        parts = parts[1:]

    patient_id = next(
        (part for part in parts if part.lower().startswith("patient")),
        None,
    )
    study_id = next(
        (part for part in parts if part.lower().startswith("study")),
        None,
    )

    return parts, patient_id, study_id


def _resolve_chexpert_image(path_to_image):
    parts, patient_id, study_id = _parse_chexpert_path(path_to_image)

    if not patient_id or not study_id:
        return None

    relative_path = Path(*parts)
    candidates = [
        CHEXPERT_PLUS_IMAGE_DIR / relative_path,
        (CHEXPERT_PLUS_IMAGE_DIR / relative_path).with_suffix(".png"),
        (CHEXPERT_PLUS_IMAGE_DIR / relative_path).with_suffix(".jpg"),
        (CHEXPERT_PLUS_IMAGE_DIR / relative_path).with_suffix(".jpeg"),
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    image_name = Path(path_to_image).with_suffix(".png").name
    study_dir = CHEXPERT_PLUS_IMAGE_DIR / patient_id / study_id

    if study_dir.exists():
        matches = list(study_dir.glob(image_name))
        if matches:
            return matches[0]

    return None


def _radgraph_text(entry):
    if not isinstance(entry, dict) or len(entry) == 0:
        return ""

    section = next(iter(entry.values()))

    if not isinstance(section, dict):
        return ""

    return _normalize_chexpert_text(section.get("text", ""))


def _load_radgraph_texts(path):
    print(f"Loading RadGraph text: {path.name}", flush=True)

    with open(path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return [
        _radgraph_text(entry)
        for entry in data
    ]


chexpert_plus_studies = {}

with open(CHEXPERT_PLUS_LABEL_FILE, "r", encoding="utf-8") as label_file:
    for row_index, line in enumerate(label_file):
        line = line.strip()

        if not line:
            continue

        row = json.loads(line)
        path_to_image = row.get("path_to_image", "")
        image_path = _resolve_chexpert_image(path_to_image)

        if image_path is None:
            continue

        _, patient_id, study_id = _parse_chexpert_path(path_to_image)
        study_key = (patient_id, study_id)

        study = chexpert_plus_studies.setdefault(
            study_key,
            {
                "patient_id": patient_id,
                "study_id": study_id,
                "radgraph_index": row_index,
                "image_paths": [],
                "label_rows": [],
            },
        )

        study["image_paths"].append(str(image_path))
        study["label_rows"].append(row)

print(
    f"Resolved {len(chexpert_plus_studies)} CheXpert Plus studies "
    "with local images.",
    flush=True,
)

findings_texts = _load_radgraph_texts(CHEXPERT_PLUS_FINDINGS_RADGRAPH)
impression_texts = _load_radgraph_texts(CHEXPERT_PLUS_IMPRESSION_RADGRAPH)

patient_to_studies = {}

for study in chexpert_plus_studies.values():
    idx = study["radgraph_index"]
    findings = findings_texts[idx] if idx < len(findings_texts) else ""
    impression = impression_texts[idx] if idx < len(impression_texts) else ""

    if not findings and not impression:
        continue

    ground_truth_parts = []

    if findings:
        ground_truth_parts.append(f"FINDINGS:\n{findings}")

    if impression:
        ground_truth_parts.append(f"IMPRESSION:\n{impression}")

    study["ground_truth"] = "\n\n".join(ground_truth_parts)
    study["image_paths"] = list(dict.fromkeys(study["image_paths"]))

    patient_to_studies.setdefault(
        study["patient_id"],
        [],
    ).append(study)

patient_ids = sorted(patient_to_studies)
random.Random(CHEXPERT_PLUS_RANDOM_SEED).shuffle(patient_ids)
patient_ids = patient_ids[:CHEXPERT_PLUS_MAX_PATIENTS]

chexpert_plus_samples = []

for patient_id in patient_ids:
    studies = sorted(
        patient_to_studies[patient_id],
        key=lambda item: item["study_id"],
    )

    chexpert_plus_samples.append(studies[0])

print(
    f"Selected {len(chexpert_plus_samples)} CheXpert Plus patients "
    f"from {len(patient_to_studies)} eligible patients."
)


def build_chexpert_plus_prompt():
    return """You are an expert thoracic radiologist AI.

Generate a concise chest X-ray report for the provided image(s).

Output ONLY:

FINDINGS:
...

IMPRESSION:
...

Rules:
- Use concise radiology-report style
- No bullet points
- No conversational text
- No explanations
""".strip()


print("Running CheXpert Plus generation and evaluation...")

chexpert_plus_outputs = []
chexpert_plus_bleu_scores = []
chexpert_plus_rouge_scores = []
chexpert_plus_f1_scores = []
chexpert_plus_times = []
chexpert_plus_num_errors = 0

for idx, sample in enumerate(chexpert_plus_samples):
    print(
        f"[{idx + 1}/{len(chexpert_plus_samples)}] "
        f"{sample['patient_id']} {sample['study_id']}"
    )

    prompt = build_chexpert_plus_prompt()
    start_time = time.time()

    generated = model.generate(
        prompt=prompt,
        image_paths=sample["image_paths"],
    )

    elapsed = time.time() - start_time

    if len(generated.strip()) == 0:
        chexpert_plus_num_errors += 1
        continue

    gt = sample["ground_truth"]

    chexpert_plus_outputs.append(
        {
            "dataset": "CheXpert_Plus_Dataset",
            "patient_id": sample["patient_id"],
            "study_id": sample["study_id"],
            "image_paths": sample["image_paths"],
            "generated_report": generated,
            "ground_truth": gt,
            "inference_time": elapsed,
        }
    )

    chexpert_plus_bleu_scores.append(
        compute_bleu(generated, gt)
    )

    chexpert_plus_rouge_scores.append(
        compute_rouge(generated, gt)
    )

    chexpert_plus_f1_scores.append(
        compute_chexpert_f1(generated, gt)
    )

    chexpert_plus_times.append(elapsed)

    cleanup_cuda()


chexpert_plus_results = {
    "dataset": "CheXpert_Plus_Dataset",
    "num_patients": len(chexpert_plus_samples),
    "num_generated_reports": len(chexpert_plus_outputs),
    "num_errors": chexpert_plus_num_errors,
    "bleu": (
        float(np.mean(chexpert_plus_bleu_scores))
        if chexpert_plus_bleu_scores
        else 0.0
    ),
    "rouge_l": (
        float(np.mean(chexpert_plus_rouge_scores))
        if chexpert_plus_rouge_scores
        else 0.0
    ),
    "chexpert_f1": (
        float(np.mean(chexpert_plus_f1_scores))
        if chexpert_plus_f1_scores
        else 0.0
    ),
    "avg_time_per_sample_s": (
        float(np.mean(chexpert_plus_times))
        if chexpert_plus_times
        else 0.0
    ),
}

print()
print("[CheXpert Plus] Results:")

for key, value in chexpert_plus_results.items():
    if isinstance(value, float):
        print(f"    {key}: {value:.4f}")
    else:
        print(f"    {key}: {value}")

with open("chexpert_plus_outputs_not_ft.jsonl", "w", encoding="utf-8") as file:
    for record in chexpert_plus_outputs:
        file.write(json.dumps(record) + "\n")

print("Saved: chexpert_plus_outputs_not_ft.jsonl")


Loading CheXpert Plus samples...
Resolved 200 CheXpert Plus studies with local images.
Loading RadGraph text: section_findings.json
Loading RadGraph text: section_impression.json
Selected 100 CheXpert Plus patients from 199 eligible patients.
Running CheXpert Plus generation and evaluation...
[1/100] patient64607 study1
[2/100] patient64650 study1
[3/100] patient64571 study1
[4/100] patient64652 study1
[5/100] patient64554 study1
[6/100] patient64583 study1
[7/100] patient64636 study1
[8/100] patient64543 study1
[9/100] patient64586 study1
[10/100] patient64605 study1
[11/100] patient64700 study1
[12/100] patient64732 study1
[13/100] patient64677 study1
[14/100] patient64684 study1
[15/100] patient64728 study1
[16/100] patient64585 study1
[17/100] patient64661 study1
[18/100] patient64717 study1
[19/100] patient64544 study1
[20/100] patient64699 study1
[21/100] patient64556 study1
[22/100] patient64686 study1
[23/100] patient64712 study1
[24/100] patient64703 study1
[25/100] patient646

In [26]:
# ============================================================
# CHEXPERT PLUS PER-PATHOLOGY EVALUATION
# Includes macro-average and micro-average across pathologies
# ============================================================

import pandas as pd
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

missing_dependencies = [
    name
    for name in [
        "chexpert_plus_outputs",
        "extract_labels",
        "PATHOLOGY_TERMS",
    ]
    if name not in globals()
]

if missing_dependencies:
    raise RuntimeError(
        "Run the CheXpert Plus generation/evaluation cell before this "
        "per-pathology cell. "
        f"Missing: {', '.join(missing_dependencies)}"
    )


def chexpert_plus_pathology_positive_vector(text, pathology):
    labels = extract_labels(text)
    return 1 if labels.get(pathology) == "positive" else 0


chexpert_plus_per_pathology_rows = []
chexpert_plus_all_y_true = []
chexpert_plus_all_y_pred = []

for pathology in PATHOLOGY_TERMS.keys():
    y_true = []
    y_pred = []

    for item in chexpert_plus_outputs:
        gt = item.get("ground_truth", "")
        pred = item.get("generated_report", "")

        if len(pred.strip()) == 0:
            continue

        true_label = chexpert_plus_pathology_positive_vector(gt, pathology)
        pred_label = chexpert_plus_pathology_positive_vector(pred, pathology)

        y_true.append(true_label)
        y_pred.append(pred_label)

    chexpert_plus_all_y_true.extend(y_true)
    chexpert_plus_all_y_pred.extend(y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    chexpert_plus_per_pathology_rows.append(
        {
            "Pathology": pathology,
            "Support_GT_Positive": int(sum(y_true)),
            "Predicted_Positive": int(sum(y_pred)),
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn),
            "Precision": float(precision),
            "Recall": float(recall),
            "F1": float(f1),
        }
    )

chexpert_plus_per_pathology_df = pd.DataFrame(
    chexpert_plus_per_pathology_rows
)

chexpert_plus_macro_precision = chexpert_plus_per_pathology_df[
    "Precision"
].mean()
chexpert_plus_macro_recall = chexpert_plus_per_pathology_df[
    "Recall"
].mean()
chexpert_plus_macro_f1 = chexpert_plus_per_pathology_df[
    "F1"
].mean()

(
    chexpert_plus_micro_precision,
    chexpert_plus_micro_recall,
    chexpert_plus_micro_f1,
    _,
) = precision_recall_fscore_support(
    chexpert_plus_all_y_true,
    chexpert_plus_all_y_pred,
    average="binary",
    zero_division=0,
)

chexpert_plus_average_rows = pd.DataFrame(
    [
        {
            "Pathology": "Macro-average",
            "Support_GT_Positive": int(
                chexpert_plus_per_pathology_df[
                    "Support_GT_Positive"
                ].sum()
            ),
            "Predicted_Positive": int(
                chexpert_plus_per_pathology_df[
                    "Predicted_Positive"
                ].sum()
            ),
            "TP": int(chexpert_plus_per_pathology_df["TP"].sum()),
            "FP": int(chexpert_plus_per_pathology_df["FP"].sum()),
            "FN": int(chexpert_plus_per_pathology_df["FN"].sum()),
            "TN": int(chexpert_plus_per_pathology_df["TN"].sum()),
            "Precision": float(chexpert_plus_macro_precision),
            "Recall": float(chexpert_plus_macro_recall),
            "F1": float(chexpert_plus_macro_f1),
        },
        {
            "Pathology": "Micro-average",
            "Support_GT_Positive": int(sum(chexpert_plus_all_y_true)),
            "Predicted_Positive": int(sum(chexpert_plus_all_y_pred)),
            "TP": int(chexpert_plus_per_pathology_df["TP"].sum()),
            "FP": int(chexpert_plus_per_pathology_df["FP"].sum()),
            "FN": int(chexpert_plus_per_pathology_df["FN"].sum()),
            "TN": int(chexpert_plus_per_pathology_df["TN"].sum()),
            "Precision": float(chexpert_plus_micro_precision),
            "Recall": float(chexpert_plus_micro_recall),
            "F1": float(chexpert_plus_micro_f1),
        },
    ]
)

chexpert_plus_per_pathology_df = (
    chexpert_plus_per_pathology_df.sort_values(
        by=["Support_GT_Positive", "F1"],
        ascending=[False, False],
    )
)

chexpert_plus_per_pathology_with_avg_df = pd.concat(
    [
        chexpert_plus_per_pathology_df,
        chexpert_plus_average_rows,
    ],
    ignore_index=True,
)

display(chexpert_plus_per_pathology_with_avg_df)

print("\n[CheXpert Plus] Per-pathology results with macro/micro averages:")
print(
    chexpert_plus_per_pathology_with_avg_df.to_string(
        index=False
    )
)

chexpert_plus_per_pathology_df.to_csv(
    "chexpert_plus_per_pathology_results.csv",
    index=False,
)

chexpert_plus_per_pathology_with_avg_df.to_csv(
    "chexpert_plus_per_pathology_results_with_avg.csv",
    index=False,
)

print("Saved: chexpert_plus_per_pathology_results.csv")
print("Saved: chexpert_plus_per_pathology_results_with_avg.csv")


,Pathology,Support_GT_Positive,Predicted_Positive,TP,FP,FN,TN,Precision,Recall,F1
0,Pleural Effusion,44,19,6,13,38,43,0.315789,0.136364,0.190476
1,Support Devices,42,2,1,1,41,57,0.500000,0.023810,0.045455
2,Atelectasis,30,0,0,0,30,70,0.000000,0.000000,0.000000
3,No Finding,28,69,22,47,6,25,0.318841,0.785714,0.453608
4,Lung Opacity,28,18,5,13,23,59,0.277778,0.178571,0.217391
5,Edema,27,8,2,6,25,67,0.250000,0.074074,0.114286
6,Consolidation,23,6,2,4,21,73,0.333333,0.086957,0.137931
7,Cardiomegaly,12,66,10,56,2,32,0.151515,0.833333,0.256410
8,Pneumothorax,12,1,0,1,12,87,0.000000,0.000000,0.000000
9,Fracture,6,7,3,4,3,90,0.428571,0.500000,0.461538



[CheXpert Plus] Per-pathology results with macro/micro averages:
                 Pathology  Support_GT_Positive  Predicted_Positive  TP  FP  FN  TN  Precision   Recall       F1
          Pleural Effusion                   44                  19   6  13  38  43   0.315789 0.136364 0.190476
           Support Devices                   42                   2   1   1  41  57   0.500000 0.023810 0.045455
               Atelectasis                   30                   0   0   0  30  70   0.000000 0.000000 0.000000
                No Finding                   28                  69  22  47   6  25   0.318841 0.785714 0.453608
              Lung Opacity                   28                  18   5  13  23  59   0.277778 0.178571 0.217391
                     Edema                   27                   8   2   6  25  67   0.250000 0.074074 0.114286
             Consolidation                   23                   6   2   4  21  73   0.333333 0.086957 0.137931
              Cardiomegaly    